In [13]:
import pandas as pd
import statistics
import math
from collections import defaultdict
import statbotics
import requests
import json
from pathlib import Path

match_json_path = Path("betterSB/orwilmatchmath.json")
match_data = json.loads(match_json_path.read_text())

sb = statbotics.Statbotics()
try:
    score_sd = sb.get_year(2026, fields=["score_sd"])
except Exception:
    score_sd = {"score_sd": None}

k = -5/8
print(score_sd)


{'score_sd': 93.78}


In [14]:

def calc_win_odds(k, score_sd_value, red_score, blue_score):
    norm_diff = (red_score - blue_score) / score_sd_value if score_sd_value else 0.0
    odds = 1 / (1 + 10**(k * norm_diff))
    return odds


def get_team_total_points(team):
    if not team:
        return 0.0
    points = team.get("points", {})
    metric = team.get("selected_metric")
    if metric:
        total_points = points.get(metric, {}).get("total_points")
        if total_points is not None:
            return float(total_points)
    fallback = team.get("selected_total")
    if fallback is None:
        fallback = team.get("selected_value", 0.0)
    try:
        return float(fallback)
    except (TypeError, ValueError):
        return 0.0


def get_alliance_team_keys(match, alliance):
    return [team.get("team_key") for team in match.get("teams", []) if team.get("alliance") == alliance]


def fetch_tba_scores(match_key):
    url = f"https://www.thebluealliance.com/api/v3/match/{match_key}"
    headers = {"X-TBA-Auth-Key": "uqTThWSrIgK7D7M3ct9fnwfIrj9m7ZzuCjwsgWsHzMtRl2xRNIm8pEQXVhfwOsBv"}
    try:
        resp = requests.get(url, headers=headers, timeout=15)
        if resp.status_code != 200:
            return None, None
        data = resp.json()
        alliances = data.get("alliances") or {}
        red = alliances.get("red") or {}
        blue = alliances.get("blue") or {}
        return red.get("score"), blue.get("score")
    except Exception:
        return None, None


def team_prior_average(prior_selected, team_key):
    if not team_key:
        return 0.0
    values = prior_selected.get(team_key, [])
    if not values:
        return 0.0
    recent_values = values[-4:]
    return sum(recent_values) / len(recent_values)


def team_prior_std(prior_selected, team_key, window=4):
    values = prior_selected.get(team_key, [])[-window:]
    if not values:
        return 0.0
    return statistics.pstdev(values)



In [15]:
sorted_matches = sorted(
    match_data.get("matches", []),
    key=lambda match: (
        match.get("set_number", 0),
        match.get("match_number", 0),
        match.get("match_key", ""),
    ),
)


In [16]:
score_sd_fallback = score_sd.get("score_sd")
processed_matches = set()
prior_selected = defaultdict(list)

def describe_team_prior(team_key):
    values = prior_selected.get(team_key, [])
    avg = team_prior_average(prior_selected, team_key)
    return values, avg
rows = []
logged = False
for match in sorted_matches:
    match_key = match.get("match_key")
    if not match_key or match_key in processed_matches:
        continue
    processed_matches.add(match_key)

    red_team_keys = get_alliance_team_keys(match, "red")[:3]
    blue_team_keys = get_alliance_team_keys(match, "blue")[:3]

    red_score = sum(team_prior_average(prior_selected, team) for team in red_team_keys)
    blue_score = sum(team_prior_average(prior_selected, team) for team in blue_team_keys)

    red_std_sq = sum(team_prior_std(prior_selected, team)**2 for team in red_team_keys)
    blue_std_sq = sum(team_prior_std(prior_selected, team)**2 for team in blue_team_keys)
    match_score_sd = math.sqrt(red_std_sq + blue_std_sq)
    score_sd_value = match_score_sd or score_sd_fallback or 1.0

    odds = calc_win_odds(k, score_sd_value, red_score, blue_score)

    if match.get("match_number") == 29 and not logged:
        logged = True
        print("Match 29 predicted averages per team:")
        for label, keys in [("red", red_team_keys), ("blue", blue_team_keys)]:
            print(f"  {label.capitalize()} alliance teams: {keys}")
            for team_key in keys:
                values, avg = describe_team_prior(team_key)
                print(f"    {team_key}: values={values}, average={avg:.2f}")
        print("  Totals: red_score=", red_score, "blue_score=", blue_score)

    r1_key = red_team_keys[0] if len(red_team_keys) > 0 else None
    r2_key = red_team_keys[1] if len(red_team_keys) > 1 else None
    r3_key = red_team_keys[2] if len(red_team_keys) > 2 else None
    b1_key = blue_team_keys[0] if len(blue_team_keys) > 0 else None
    b2_key = blue_team_keys[1] if len(blue_team_keys) > 1 else None
    b3_key = blue_team_keys[2] if len(blue_team_keys) > 2 else None

    alliances = match.get("alliances", {})
    red_pred = alliances.get("red", {}).get("selected_total")
    blue_pred = alliances.get("blue", {}).get("selected_total")
    sb_pred_diff = None
    if red_pred is not None and blue_pred is not None:
        sb_pred_diff = red_pred - blue_pred

    ba_red_score, ba_blue_score = fetch_tba_scores(match_key)
    if ba_red_score is None or ba_blue_score is None:
        correct_pred = None
    else:
        actual_red_win = ba_red_score > ba_blue_score
        predicted_red_win = odds >= 0.5
        correct_pred = int(actual_red_win == predicted_red_win)

    rows.append({
        "match_number": int(match.get("match_number", 0)),
        "r1": r1_key,
        "r2": r2_key,
        "r3": r3_key,
        "b1": b1_key,
        "b2": b2_key,
        "b3": b3_key,
        "red_score": red_score,
        "blue_score": blue_score,
        "red_win_odds": odds,
        "ba_redScore": ba_red_score,
        "ba_blueScore": ba_blue_score,
        "correct_pred": correct_pred,
        "sb_red_pred": red_pred,
        "sb_blue_pred": blue_pred,
        "sb_pred_diff": sb_pred_diff,
    })

    team_lookup = {
        team.get("team_key"): team for team in match.get("teams", []) if team.get("team_key")
    }

    for team_key in red_team_keys + blue_team_keys:
        value = get_team_total_points(team_lookup.get(team_key))
        prior_selected[team_key].append(value)

preds_df = pd.DataFrame(rows)
preds_df.to_csv("preds.csv", index=False)
preds_df.head()



Match 29 predicted averages per team:
  Red alliance teams: ['frc3574', 'frc2550', 'frc3673']
    frc3574: values=[40.38, 0.0, 0.0, 0.0, 15.75], average=3.94
    frc2550: values=[37.5, 34.964749999999995, 16.7625, 60.158750000000005, 22.588], average=33.62
    frc3673: values=[6.18, 14.32, 0.0, 14.18, 56.211999999999996], average=21.18
  Blue alliance teams: ['frc5937', 'frc957', 'frc7034']
    frc5937: values=[106.26527, 59.9, 48.64, 45.36, 135.32292500000003], average=72.31
    frc957: values=[54.739999999999995, 56.79, 88.92107000000001, 69.17, 56.166725], average=67.76
    frc7034: values=[41.379999999999995, 40.93, 118.239, 57.63, 43.61], average=65.10
  Totals: red_score= 58.733999999999995 blue_score= 205.16993000000002


,match_number,r1,r2,r3,b1,b2,b3,red_score,blue_score,red_win_odds,ba_redScore,ba_blueScore,correct_pred,sb_red_pred,sb_blue_pred,sb_pred_diff
0,1,frc2635,frc3673,frc3024,frc10444,frc2374,frc6831,0.0,0.0,0.5,52,131,0,51.992,131.057,-79.065
1,2,frc847,frc2898,frc4043,frc4488,frc1540,frc2550,0.0,0.0,0.5,58,300,0,57.963,229.760,-171.797
2,3,frc3574,frc3663,frc5970,frc6696,frc3636,frc5975,0.0,0.0,0.5,55,18,1,87.827,17.795,70.032
3,4,frc9613,frc6343,frc4662,frc1425,frc4513,frc5937,0.0,0.0,0.5,50,215,0,50.010,215.096,-165.086
4,5,frc7034,frc2915,frc1595,frc2990,frc9600,frc9438,0.0,0.0,0.5,73,65,1,72.997,65.020,7.977


In [ ]:
from math import sqrt


def compute_rmse(pred_series, truth_series):
    delta = pred_series - truth_series
    return sqrt((delta**2).mean()) if not delta.empty else float('nan')


def actual_red_win(mask):
    subset = preds_df.loc[mask]
    return (
        subset['ba_redScore'] > subset['ba_blueScore']
    ).astype(float)


def diff_to_odds(diff_series):
    return 1 / (1 + 10**(k * diff_series))

statbotics_red_mask = (
    preds_df['sb_red_pred'].notna()
    & preds_df['ba_redScore'].notna()
)
statbotics_blue_mask = (
    preds_df['sb_blue_pred'].notna()
    & preds_df['ba_blueScore'].notna()
)
statbotics_diff_mask = (
    preds_df['sb_pred_diff'].notna()
    & preds_df['ba_redScore'].notna()
    & preds_df['ba_blueScore'].notna()
)

statbotics_red_rmse = compute_rmse(
    preds_df.loc[statbotics_red_mask, 'sb_red_pred'],
    preds_df.loc[statbotics_red_mask, 'ba_redScore'],
)
statbotics_blue_rmse = compute_rmse(
    preds_df.loc[statbotics_blue_mask, 'sb_blue_pred'],
    preds_df.loc[statbotics_blue_mask, 'ba_blueScore'],
)
statbotics_diff_rmse = compute_rmse(
    preds_df.loc[statbotics_diff_mask, 'sb_pred_diff'],
    preds_df.loc[statbotics_diff_mask, 'ba_redScore']
    - preds_df.loc[statbotics_diff_mask, 'ba_blueScore'],
)

statbotics_win_rmse = float('nan')
statbotics_win_mask = statbotics_diff_mask
if statbotics_win_mask.any():
    statbotics_win_rmse = compute_rmse(
        diff_to_odds(preds_df.loc[statbotics_win_mask, 'sb_pred_diff']),
        actual_red_win(statbotics_win_mask),
    )

calc_red_mask = (
    preds_df['red_score'].notna()
    & preds_df['ba_redScore'].notna()
)
calc_blue_mask = (
    preds_df['blue_score'].notna()
    & preds_df['ba_blueScore'].notna()
)
calc_diff_mask = (
    preds_df['red_score'].notna()
    & preds_df['blue_score'].notna()
    & preds_df['ba_redScore'].notna()
    & preds_df['ba_blueScore'].notna()
)

calc_red_rmse = compute_rmse(
    preds_df.loc[calc_red_mask, 'red_score'],
    preds_df.loc[calc_red_mask, 'ba_redScore'],
)
calc_blue_rmse = compute_rmse(
    preds_df.loc[calc_blue_mask, 'blue_score'],
    preds_df.loc[calc_blue_mask, 'ba_blueScore'],
)
calc_diff_rmse = compute_rmse(
    preds_df.loc[calc_diff_mask, 'red_score']
    - preds_df.loc[calc_diff_mask, 'blue_score'],
    preds_df.loc[calc_diff_mask, 'ba_redScore']
    - preds_df.loc[calc_diff_mask, 'ba_blueScore'],
)

odds_mask = (
    preds_df['red_win_odds'].notna()
    & preds_df['ba_redScore'].notna()
    & preds_df['ba_blueScore'].notna()
)
odds_rmse = float('nan')
if odds_mask.any():
    odds_rmse = compute_rmse(
        preds_df.loc[odds_mask, 'red_win_odds'],
        actual_red_win(odds_mask),
    )

print(f"RMSE Statbotics red score: {statbotics_red_rmse:.3f}")
print(f"RMSE Statbotics blue score: {statbotics_blue_rmse:.3f}")
print(f"RMSE Statbotics score diff: {statbotics_diff_rmse:.3f}")
print(f"RMSE Statbotics win odds: {statbotics_win_rmse:.3f}")
print(f"RMSE calc red score: {calc_red_rmse:.3f}")
print(f"RMSE calc blue score: {calc_blue_rmse:.3f}")
print(f"RMSE calc score diff: {calc_diff_rmse:.3f}")
print(f"RMSE red win odds vs actual outcome: {odds_rmse:.3f}")


RMSE Statbotics red score: 15.095
RMSE Statbotics blue score: 14.948
RMSE Statbotics score diff: 20.668
RMSE Statbotics win odds: 0.177
RMSE calc red score: 63.625
RMSE calc blue score: 73.871
RMSE calc score diff: 86.397
RMSE red win odds vs actual outcome: 0.413
Total absolute difference in Statbotics predictions: 6325.298
